# BERTopic

Ajusta BERTopic y reduce la partición a seis tópicos, en la corrida de referencia y sobre diez semillas.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Corrida exploratoria y corrida de referencia (Tablas 4.3 y 4.4)

In [ ]:
# BERTOPIC — TOPIC MODELING



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import os
import unicodedata
from umap import UMAP
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import pickle

nltk.download('stopwords', quiet=True)

# 0. UTILIDAD: NORMALIZACIÓN DE TILDES

def remove_accents(text):
    if not isinstance(text, str):
        return text
    return ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')

# ============================================================
# 1. CARGA DE DATOS
# ============================================================
df = pd.read_pickle("../df_generic_cleaned.pkl")
print(f"Reseñas cargadas: {len(df)}")


# ============================================================
# 2. STOPWORDS — LISTAS EXACTAS DE WORD2VEC
# ============================================================
CONTEXTO = {
    'lugar', 'sitio', 'espacio', 'zona', 'parte', 'parque', 'montevideo', 'uruguay',
    'visita', 'visitar', 'paseo', 'pasear', 'ir', 'hacer', 'dia', 'noche',
    'fin', 'semana', 'año', 'vez', 'momento', 'cosa', 'caso', 'tema', 'tipo',
    'aunque', 'sino', 'pues', 'tal', 'tan', 'asi', 'ahi', 'alla', 'aca',
    'pais', 'ciudad', 'capital', 'gente', 'persona', 'publico', 'mate', 'pasar',
    'quedar', 'estar', 'ser', 'haber', 'tener', 'poder', 'deber',
    'dar', 'ver', 'saber', 'querer', 'llegar', 'decir', 'salir', 'venir',
    'encontrar', 'dejar', 'poner', 'parecer', 'llevar', 'seguir',
    'etc', 'siempre', 'alguno', 'solo', 'medio', 'ahora', 'mismo',
}
SENTIMIENTO = {
    'lindo', 'hermoso', 'bonito', 'bello', 'precioso',
    'feo', 'horrible', 'espantoso', 'terrible',
    'bueno', 'malo', 'buen',
    'excelente', 'espectacular', 'genial', 'fantastico', 'increible',
    'maravilloso', 'divino', 'impecable', 'impresionante',
    'pesimo', 'lamentable', 'desastre', 'patetico',
    'mejor', 'peor', 'mayor', 'menor', 'gran', 'grande',
    'bien', 'mal', 'super', 'muy', 'bastante', 'poco', 'mucho',
    'agradable', 'desagradable', 'recomendable', 'recomendado',
    'tranquilo', 'paz', 'relax',
    'gustar', 'encantar', 'disfrutar', 'recomendar',
    'valer', 'ideal', 'perfecto', 'destacar', 'pena', 'disfrutable',
}
ERRORES = {
    'lindar', 'exelente', 'exelentes', 'escelente',
    'imprecionante', 'ermoso', 'ermosa',
    'recomeir', 'recomiendable', 'recomendacion', 'mas',
}
CORTESIA = {'gracias'}
LUGARES  = {'prado'}

# Ajuste de sintaxis: se usan llaves para definir el set
PROTEGER = {'nino', 'ninos', 'niño', 'niña', 'niñas', 'niños'}  

# Nombres de parques
nombres_parque = set()
for nombre in df['nombre_parque'].str.lower().unique():
    for token in nombre.split():
        nombres_parque.add(remove_accents(token))
nombres_parque -= PROTEGER

# Construcción del Set Final
final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
for grupo in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in grupo}
final_stopwords |= nombres_parque

# CountVectorizer necesita una lista, no un set
stop_list = list(final_stopwords) 
print(f"Stopwords totales: {len(stop_list)}")


# ============================================================
# 3. CONFIGURAR Y ENTRENAR BERTOPIC
# ============================================================

# remove_accents como preprocesador
vectorizer_model = CountVectorizer(
    preprocessor=remove_accents,
    stop_words=stop_list,
    min_df=3,
    ngram_range=(1, 2)
)

representation_model = KeyBERTInspired()
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
topic_model = BERTopic(
    language="spanish",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True,
    min_topic_size=15,
    nr_topics="auto",
)

print("Entrenando BERTopic...")
docs = df['text'].tolist()

topics, probs = topic_model.fit_transform(docs)
# ============================================================
# 4. RESULTADOS INICIALES (exploratorio)
# ============================================================
freq = topic_model.get_topic_info()
n_topics = len(freq[freq['Topic'] != -1])
n_outliers = (np.array(topics) == -1).sum()

print(f"\nRESULTADOS BERTOPIC — corrida exploratoria")
print(f"Tópicos encontrados con min_topic_size=15 (sin outliers): {n_topics}")
print(f"Outliers (tópico -1): {n_outliers} ({n_outliers/len(docs)*100:.1f}%)")
print(f"Top 10 tópicos por frecuencia:")
display(freq.head(11))

freq.to_csv("../Topic/Resultados/Topico/BERTOPIC_info_auto.csv", index=False)

# ============================================================
# 5. CORRIDA FINAL: min_topic_size=80 + REDUCCIÓN A 6
# ============================================================
topic_model = BERTopic(
    language="spanish",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True,
    min_topic_size=80,
)
topics, probs = topic_model.fit_transform(docs)

freq_pre_reduce = topic_model.get_topic_info()
n_topics_pre_reduce = len(freq_pre_reduce[freq_pre_reduce['Topic'] != -1])
print(f"\nTópicos naturales con min_topic_size=80 (antes de reducir): {n_topics_pre_reduce}")

# Se pasa 7 para obtener 6 tópicos reales (el ruido cuenta como uno).
topic_model.reduce_topics(docs, nr_topics=7)

topics_6 = topic_model.topics_
freq_6 = topic_model.get_topic_info()
n_outliers_6 = (np.array(topics_6) == -1).sum()
probs_6 = topic_model.approximate_distribution(docs, calculate_tokens=True)[0]

# Tópicos reales presentes (excluyendo -1)
topicos_reales = sorted([t for t in set(topics_6) if t != -1])
print(f"\nTópicos reales tras reducción: {len(topicos_reales)} → {topicos_reales}")

# ============================================================
# 6. KEYWORDS POR TÓPICO (modelo reducido)
# ============================================================
print("\nKeywords de los tópicos:")

topic_keywords = {}
for topic_id in topicos_reales:
    words = topic_model.get_topic(topic_id)
    if words:
        topic_keywords[topic_id] = words[:15]
        kw_str = ', '.join([f"{w} ({s:.3f})" for w, s in words[:10]])
        print(f"\nTópico {topic_id}: {kw_str}")

# --- Plot dinámico ----------------------------------------
# Antes: fig de 3x2 fija con range(6). Ahora se ajusta al N real.
n_real = len(topic_keywords)
ncols = 2
nrows = (n_real + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(14, 4 * nrows),
    constrained_layout=True
)
axes = np.array(axes).flatten()

for idx, topic_id in enumerate(topicos_reales):
    ax = axes[idx]
    words = topic_keywords.get(topic_id, [])[:15]
    if not words:
        ax.set_axis_off()
        continue
    labels = [w for w, s in words]
    scores = [s for w, s in words]

    ax.barh(labels[::-1], scores[::-1], color='teal')
    ax.set_title(f'Tópico {topic_id}', fontsize=12, fontweight='bold')
    ax.set_xlabel('c-TF-IDF score')

# Si sobran subplots por la grilla 2-col impar, los apagamos
for j in range(n_real, len(axes)):
    axes[j].set_axis_off()

fig.suptitle(f'Palabras clave por tópico (BERTopic, K={n_real})',
             fontsize=14, fontweight='bold')
plt.savefig("../Topic/Resultados/Topico/BERTOPIC_keywords_6_topics.png", dpi=300, bbox_inches='tight')
plt.show()


# ============================================================
# 7. VISUALIZACIONES HTML (modelo reducido)
# ============================================================
try:
    topic_model.visualize_barchart(
        top_n_topics=n_real, n_words=10
    ).write_html("../Topic/Resultados/Topico/BERTOPIC_barras_6.html")

    topic_model.visualize_topics().write_html("../Topic/Resultados/Topico/BERTOPIC_mapa_6.html")
    topic_model.visualize_heatmap().write_html("../Topic/Resultados/Topico/BERTOPIC_similitud_6.html")
    print("Gráficos HTML guardados.")
except Exception as e:
    print(f"Error generando HTML: {e}")

# ============================================================
# 8. ASIGNAR TÓPICOS AL DATAFRAME
# ============================================================
df['topico_bertopic'] = topics_6
df['topico_bertopic_nombre'] = df['topico_bertopic'].map(
    {t: f"Tópico {t}" for t in topicos_reales}
)
df.loc[df['topico_bertopic'] == -1, 'topico_bertopic_nombre'] = "Outlier"

print(f"Distribución de tópicos:")
print(df['topico_bertopic'].value_counts().sort_index())

# ============================================================
# 9. DISTRIBUCIÓN POR PARQUE
# ============================================================
perfil_parques = pd.crosstab(
    df['nombre_parque'],
    df['topico_bertopic'],
    normalize='index'
) * 100

col_rename = {t: f"Tópico {t}" for t in perfil_parques.columns if t != -1}
col_rename[-1] = "Outlier"
perfil_parques = perfil_parques.rename(columns=col_rename)

cols_sorted = [c for c in perfil_parques.columns if c != "Outlier"]
cols_sorted = sorted(cols_sorted)
if "Outlier" in perfil_parques.columns:
    cols_sorted.append("Outlier")
perfil_parques = perfil_parques[cols_sorted]

print("\nDistribución por parque (%):")
display(perfil_parques.round(1))

plt.figure(figsize=(12, 8))
sns.heatmap(
    perfil_parques,
    annot=True, fmt=".1f",
    cmap="YlGnBu", linewidths=.5,
    cbar_kws={'label': '% de reseñas'}
)
plt.title(f'Distribución de tópicos por parque (BERTopic, K={n_real})', fontsize=14)
plt.xlabel('Tópico')
plt.ylabel('Parque')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("bertopic_heatmap_6.png", dpi=150, bbox_inches='tight')
plt.show()

ax = perfil_parques.plot(
    kind='barh', stacked=True,
    figsize=(12, 8), colormap='tab10', edgecolor='white'
)
plt.title(f'Distribución de tópicos por parque (BERTopic, K={n_real})', fontsize=14)
plt.xlabel('% de reseñas')
plt.ylabel('')
plt.legend(title='Tópico', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("bertopic_barras_parques_6.png", dpi=150, bbox_inches='tight')
plt.show()


# ============================================================
# 10. RESEÑAS REPRESENTATIVAS POR TÓPICO
# ============================================================
print("RESEÑAS REPRESENTATIVAS POR TÓPICO")

for topic_id in sorted(df['topico_bertopic'].unique()):
    if topic_id == -1:
        label = "OUTLIERS"
    else:
        label = f"Tópico {topic_id}"

    subset = df[df['topico_bertopic'] == topic_id]
    n = len(subset)

    try:
        muestras = subset.sample(min(5, n), random_state=42)
    except ValueError:
        muestras = subset

    print(f"\n--- {label} ({n} reseñas, {n/len(df)*100:.1f}%) ---")
    for _, row in muestras.iterrows():
        texto = row['text'][:200]
        print(f"  [{row['nombre_parque']}] {texto}")


# ============================================================
# 11. COHERENCIA (C_v)
# ============================================================
# ============================================================
# 11. COHERENCIA (C_v)
# ============================================================
print("COHERENCIA C_v")

topics_words = []
for topic_id in topicos_reales:
    words = topic_model.get_topic(topic_id)
    if words:
        # Gensim necesita solo las palabras, no los scores
        topics_words.append([w for w, s in words[:10]])

# --- EL BLINDAJE ---
# Extraemos el "analizador" del CountVectorizer. 
# Esto tokeniza los textos usando EXACTAMENTE las mismas reglas 
# (y la misma función remove_accents) que usó BERTopic.
analyzer = vectorizer_model.build_analyzer()

tokenized_docs = [analyzer(doc) for doc in docs]
dictionary = Dictionary(tokenized_docs)
# -------------------

coherence_model = CoherenceModel(
    topics=topics_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_cv = coherence_model.get_coherence()
print(f"Coherencia C_v global: {coherence_cv:.4f}")

coherence_per_topic = coherence_model.get_coherence_per_topic()
for i, c in enumerate(coherence_per_topic):
    print(f"  Tópico {topicos_reales[i]}: {c:.4f}")


# ============================================================
# 12. SOFT CLUSTERING — DISTRIBUCIÓN DE PROBABILIDADES
# ============================================================
print("SOFT CLUSTERING — ANÁLISIS DE PROBABILIDADES")

if probs_6 is not None and len(probs_6) > 0:
    try:
        probs_array = np.array(probs_6)
        if probs_array.ndim == 2:
            from scipy.stats import entropy
            entropias = [entropy(p) for p in probs_array if p.sum() > 0]
            print(f"Entropía promedio de asignación: {np.mean(entropias):.4f}")
            print(f"Entropía mediana: {np.median(entropias):.4f}")
            print(f"(Mayor entropía = reseñas más ambiguas entre tópicos)")

            max_probs = probs_array.max(axis=1)
            print(f"\nConfianza promedio (max prob): {max_probs.mean():.4f}")
            print(f"Confianza mediana: {np.median(max_probs):.4f}")

            plt.figure(figsize=(8, 5))
            plt.hist(max_probs, bins=50, color='teal', alpha=0.7, edgecolor='white')
            plt.axvline(max_probs.mean(), color='red', linestyle='--',
                        label=f'Media: {max_probs.mean():.3f}')
            plt.title('Distribución de confianza en asignación de tópico (BERTopic)')
            plt.xlabel('Probabilidad máxima (confianza)')
            plt.ylabel('Frecuencia')
            plt.legend()
            plt.tight_layout()
            plt.savefig("../Topic/Resultados/Topico/BERTOPIC_confianza.png", dpi=300, bbox_inches='tight')
            plt.show()
        else:
            print("Probabilidades no disponibles en formato 2D.")
    except Exception as e:
        print(f"Error procesando probabilidades: {e}")
else:
    print("Probabilidades no disponibles.")

# ============================================================
# 13. RESUMEN
# ============================================================
print("RESUMEN")
print(f"Modelo de embeddings: paraphrase-multilingual (default BERTopic español)")
print(f"Tópicos naturales con min_topic_size=15 (exploratorio): {n_topics}")
print(f"Tópicos naturales con min_topic_size=80 (pre-reducción): {n_topics_pre_reduce}")
print(f"Tópicos tras reducción forzada: {n_real}")
print(f"Outliers (tópico -1) en modelo final: {n_outliers_6} ({n_outliers_6/len(docs)*100:.1f}%)")
print(f"Coherencia C_v (K={n_real}): {coherence_cv:.4f}")
print(f"min_topic_size del modelo final: 80")
print(f"Reseñas totales: {len(docs)}")


df.to_pickle("../Topic/Dataframes/df_bertopic_final.pkl")
freq_6.to_csv("../Topic/Resultados/Topico/bertopic_info_6.csv", index=False)
print("Archivos guardados.")

### Representación de los tópicos en minúsculas (Sección 3.5.2)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# preprocesador corregido: minúscula + sin acentos (antes solo sacaba acentos)
vectorizer_fix = CountVectorizer(
    preprocessor=lambda t: remove_accents(t).lower(),
    stop_words=stop_list,
    min_df=3,
    ngram_range=(1, 2),
)
topic_model.update_topics(docs, vectorizer_model=vectorizer_fix, representation_model=None)

for t in [0, 1, 2, 3, 4, 5]:
    palabras = [w for w, _ in topic_model.get_topic(t)][:10]
    print(f"Tópico {t}: {', '.join(palabras)}")

### Palabras y corpus de referencia de la corrida de referencia

In [ ]:
import pickle
analyzer = topic_model.vectorizer_model.build_analyzer()
info = topic_model.get_topic_info()
topicos_reales = [t for t in info['Topic'].tolist() if t != -1]
topic_words_raw = [[w for w, s in topic_model.get_topic(t)][:100] for t in topicos_reales]
ref_texts = [analyzer(doc) for doc in docs]
pickle.dump({'topic_words_raw': topic_words_raw, 'ref_texts': ref_texts, 'final_stopwords': final_stopwords},
            open("../Topic/Dataframes/ckpt_bertopic.pkl", "wb"))
print("ckpt_bertopic.pkl OK | tópicos:", len(topic_words_raw)) 

### Palabras clave por tópico

In [ ]:
import matplotlib.pyplot as plt

topics = [0, 1, 2, 3, 4, 5]
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, t in zip(axes.flat, topics):
    pares = topic_model.get_topic(t)          # [(palabra, c-TF-IDF score), ...]
    palabras = [w for w, _ in pares][::-1]
    scores   = [s for _, s in pares][::-1]
    ax.barh(palabras, scores, color='teal')
    ax.set_title(f"Tópico {t}")
    ax.set_xlabel("c-TF-IDF score")
plt.suptitle("Palabras clave por tópico — BERTopic (representación filtrada)", fontsize=14)
plt.tight_layout()
plt.savefig("BERTOPIC_keywords_FILTRADO.png", dpi=150, bbox_inches="tight")
plt.show()
print("guardado: BERTOPIC_keywords_FILTRADO.png")

### Diez semillas (Tablas 3.2, 4.1 y 4.2)

In [ ]:
# ============================================================
# MULTI-SEMILLA — BERTopic  (autocontenido)
# BERTopic no tiene barrido de K (el nro emerge de HDBSCAN):
# se reporta el nro de topicos naturales por semilla (estabilidad de granularidad,
# Hoyle 2022) y se captura el top-100 tras reducir a 6 (tabla de calidad).
# El embedding es DETERMINISTA -> se calcula UNA vez y se cachea; solo varia UMAP.
# No se descartan semillas.
# ============================================================
import pickle, os, unicodedata, numpy as np, pandas as pd
from umap import UMAP
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"
SEEDS = [42, 0, 1, 2, 3, 4, 5, 6, 7, 8]

def remove_accents(text):
    if not isinstance(text, str): return text
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

df = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
docs = df['text'].tolist()

CONTEXTO = {
    'lugar','sitio','espacio','zona','parte','parque','montevideo','uruguay',
    'visita','visitar','paseo','pasear','ir','hacer','dia','noche',
    'fin','semana','año','vez','momento','cosa','caso','tema','tipo',
    'aunque','sino','pues','tal','tan','asi','ahi','alla','aca',
    'pais','ciudad','capital','gente','persona','publico','mate','pasar',
    'quedar','estar','ser','haber','tener','poder','deber',
    'dar','ver','saber','querer','llegar','decir','salir','venir',
    'encontrar','dejar','poner','parecer','llevar','seguir',
    'etc','siempre','alguno','solo','medio','ahora','mismo'}
SENTIMIENTO = {
    'lindo','hermoso','bonito','bello','precioso','feo','horrible','espantoso','terrible',
    'bueno','malo','buen','excelente','espectacular','genial','fantastico','increible',
    'maravilloso','divino','impecable','impresionante','pesimo','lamentable','desastre','patetico',
    'mejor','peor','mayor','menor','gran','grande','bien','mal','super','muy','bastante','poco','mucho',
    'agradable','desagradable','recomendable','recomendado','tranquilo','paz','relax',
    'gustar','encantar','disfrutar','recomendar','valer','ideal','perfecto','destacar','pena','disfrutable'}
ERRORES = {'lindar','exelente','exelentes','escelente','imprecionante','ermoso','ermosa',
    'recomeir','recomiendable','recomendacion','mas'}
CORTESIA = {'gracias'}; LUGARES = {'prado'}
PROTEGER = {'nino','ninos','niño','niña','niñas','niños'}
nombres = set()
for nombre in df['nombre_parque'].str.lower().unique():
    for t in nombre.split(): nombres.add(remove_accents(t))
nombres -= PROTEGER
final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
for g in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in g}
final_stopwords |= nombres
stop_list = list(final_stopwords)

# embeddings: DETERMINISTAS -> una sola vez (cache)
EMB = f"{DFDIR}/prep_bertopic_emb.npy"
emb_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
if os.path.exists(EMB):
    embeddings = np.load(EMB); print(f"embeddings cargados {embeddings.shape}")
else:
    embeddings = emb_model.encode(docs, show_progress_bar=True)
    np.save(EMB, embeddings); print(f"embeddings creados {embeddings.shape}")

vectorizer_base = CountVectorizer(preprocessor=remove_accents, stop_words=stop_list, min_df=3, ngram_range=(1, 2))
vectorizer_fix  = CountVectorizer(preprocessor=lambda t: remove_accents(t).lower(), stop_words=stop_list, min_df=3, ngram_range=(1, 2))
analyzer = vectorizer_fix.build_analyzer()
ref_texts = [analyzer(doc) for doc in docs]   # invariante a la semilla

topw_by_seed = {}; n_topics_by_seed = {}
for s in SEEDS:
    umap_s = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=s)
    tm = BERTopic(language="spanish", embedding_model=emb_model, vectorizer_model=vectorizer_base,
                  umap_model=umap_s, representation_model=KeyBERTInspired(),
                  calculate_probabilities=False, verbose=False, min_topic_size=80)
    tm.fit_transform(docs, embeddings=embeddings)
    n_nat = len([t for t in tm.get_topic_info()['Topic'].tolist() if t != -1])
    if n_nat > 6:
        tm.reduce_topics(docs, nr_topics=7)
    tm.update_topics(docs, vectorizer_model=vectorizer_fix, representation_model=None)
    reales = [t for t in tm.get_topic_info()['Topic'].tolist() if t != -1]
    topw_by_seed[s] = [[w for w, _ in tm.get_topic(t)][:100] for t in reales]
    n_topics_by_seed[s] = n_nat   # naturales antes de reducir (granularidad)
    print(f"BERTopic seed={s:>2} | naturales={n_nat} | reducidos={len(reales)}")

pickle.dump({'topic_words_raw_by_seed': topw_by_seed, 'n_topics_by_seed': n_topics_by_seed,
             'ref_texts': ref_texts, 'final_stopwords': final_stopwords, 'seeds': SEEDS},
            open(f"{DFDIR}/ckpt_bertopic_multiseed.pkl", "wb"))
print(f"OK -> ckpt_bertopic_multiseed.pkl | naturales por semilla: {n_topics_by_seed}")